# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library, starting from the Croissant schema and exploring the record sets and fields programmatically by their `@id`.

### Dataset Source
Dataset URL: https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load Croissant metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"Dataset Title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Let's review available record sets, fields, and their `@id` values for accurate future referencing.

In [ ]:
# Show the available record sets and their IDs

def get_record_sets_info(dataset):
    # `dataset.record_sets` provides a list of RecordSet objects
    print("Available record sets:")
    record_sets_info = []
    for rec in dataset.record_sets:
        info = {
            '@id': rec.id,
            'name': rec.name,
            'description': getattr(rec, 'description', None),
            'fields': [(field.id, field.name) for field in rec.fields]
        }
        record_sets_info.append(info)
        print(f"- @id: {rec.id} | name: {rec.name}")
        for field in rec.fields:
            print(f"    - field @id: {field.id} | name: {field.name} | type: {field.data_type}")
    return record_sets_info

record_sets_info = get_record_sets_info(dataset)

# Obtain a list of record set ids for programmatic use
record_set_ids = [rs['@id'] for rs in record_sets_info]

## 3. Data Extraction
We will load the data from the main record set into a DataFrame, referencing all entities by their `@id`. Typically, Croissant datasets of this type have one main tabular record set (e.g., protein table) – use the first record set for demonstration.

You can programmatically choose which record set to analyze if multiple exist.

In [ ]:
# Load data from each record set into a dictionary of DataFrames, keyed by RecordSet @id
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# For demonstration, select the first record set for deeper exploration
main_record_set_id = record_set_ids[0]
print(f"\nMain Record Set: {main_record_set_id}")
print("Available columns/fields:")
print(dataframes[main_record_set_id].columns.tolist())
display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's process and explore the data using fields referenced by their `@id` only.

- Select a numeric field for filtering/normalizing (e.g., coverage, abundance, MW – check available columns above).
- Perform a simple filter, normalization, and group-by operation (if suitable fields exist).

*Update the chosen field `@id` and group-by field `@id` as found in your record set overview above for robust referencing.*

In [ ]:
# For demonstration, attempt to find reasonable numeric and grouping fields by @id

df = dataframes[main_record_set_id]

# Try to choose a numeric field by checking dtypes or known column names:
possible_numeric_fields = [col for col in df.columns if df[col].dtype in [int, float] or pd.api.types.is_numeric_dtype(df[col])]
print("Numeric fields detected:", possible_numeric_fields)

# For the sake of the example, let us select the first numeric field (usually 'coverage', 'abundance', or 'MW')
if possible_numeric_fields:
    numeric_field_id = possible_numeric_fields[0]  # Use actual @id of the column
else:
    numeric_field_id = df.columns[0]  # Fallback

print(f"Working with numeric field (by @id): {numeric_field_id}\n")

# Filter values above a threshold (choose an example threshold)
threshold = 10
filtered_df = df.copy()
if pd.api.types.is_numeric_dtype(filtered_df[numeric_field_id]):
    filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold} (showing up to 5 rows):")
    display(filtered_df.head())
else:
    print(f"The field {numeric_field_id} is not numeric, skipping filtering.")

# Normalize numeric field
if pd.api.types.is_numeric_dtype(filtered_df[numeric_field_id]):
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} field (showing up to 5 rows):")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by a categorical field
possible_group_fields = [col for col in df.columns if df[col].dtype == 'object' and col != numeric_field_id]
if possible_group_fields:
    group_field_id = possible_group_fields[0]
    print(f"\nGrouping by field (by @id): {group_field_id}")
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Mean {numeric_field_id} by {group_field_id} (top 5):")
    display(grouped_df.head())
else:
    print("No categorical field found for grouping.")

## 5. Visualization
Visualize the distribution of the selected numeric field and the normalized values. If grouped data exists, visualize group means as well.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id], bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    
# If normalized field exists
if f"{numeric_field_id}_normalized" in filtered_df.columns:
    plt.figure(figsize=(7, 4))
    sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], bins=30, kde=True)
    plt.title(f'Normalized Distribution of {numeric_field_id}')
    plt.xlabel(f"{numeric_field_id} (normalized)")
    plt.ylabel('Count')
    plt.show()

# If group statistics exist, plot bar
if 'grouped_df' in locals():
    plt.figure(figsize=(10, 5))
    sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id)
    plt.title(f'Mean {numeric_field_id} by {group_field_id}')
    plt.xlabel(group_field_id)
    plt.ylabel(f'Mean {numeric_field_id}')
    plt.xticks(rotation=45, ha='right')
    plt.show()

## 6. Conclusion
- The FAIR² dataset was loaded using its Croissant metadata via the `mlcroissant` library.
- We explored available record sets, identified fields by their `@id`, and loaded the primary record set as a DataFrame.
- Simple EDA revealed basic statistics and demonstrated filtering, normalization, and grouping – all referencing fields by their Croissant `@id` as best practice for FAIR data handling.
- The dataset is now ready for downstream domain analysis and model development.